# Fine-tuning Qwen3.5 on survey data with Unsloth: the `v1` recipe, for the final adapter

Trains a LoRA adapter so the model predicts **how people answer survey questions**, from six public sources in one format, exactly as the first training notebook did. This version applies what the held-out tests found.

| | First run | This notebook |
|---|---|---|
| LoRA targets | `q/k/v/o`, `gate/up/down`: only the 6 full-attention layers and the MLPs (6,389,760 trainable parameters) | **every Linear in the language layers, including the 18 layers' linear-attention projections** (`in_proj_qkv`, `in_proj_z`, `in_proj_b`, `in_proj_a`, `out_proj`): 10,822,656 trainable parameters at rank 16 |
| WVS / Afrobarometer profiles | wrong columns picked (`sex: Never justifiable`, `education: Don't belong`, `gender: Black / African`, ...) | columns accepted only if their answer labels fit the role; an **audit** prints what was chosen; Afrobarometer education is pinned to `Q94` (the earlier run's `Q97` is "Perceived survey sponsor") |
| Training length | fixed 300 steps | a **time budget** converted into steps from a measured step time, with a correction for the measured-vs-real gap (8.0 s measured, about 10.3 s real on a T4) |
| Datasets | all six | all six. **WVS is included**: the held-out test left it out only to measure generalisation to an unseen survey |
| Safety | none | optional checkpoints every N steps, and resume after a crash or a disconnected session |

**What the tests behind this showed** (T4, 373 steps each, held-out respondents, log loss, lower is better): attaching the linear-attention layers was nominally best of the four variants tried (Afrobarometer 1.187 vs 1.199, WVS 1.299 vs 1.331 for the old targets), rank 64 and heavier individual-level weighting were not better, and run length was far shorter than an epoch. Those differences are small and had no confidence intervals: this is a direction, not a proven gain. What the leaderboard showed (your report): training the LoRA raised the score from 0.42 to 0.46.

**Honest status**
* The code was tested on tiny random models on CPU, including a tiny Qwen3.5-style hybrid, and the earlier version of this recipe ran on your T4. **This exact combination (the 10.8M-parameter targets, the time-budget step count and the checkpoint/resume cells) has not run end to end on the real model.** The training cell prints how many LoRA modules landed on each type: **check that `linear_attn.*` appear** (they did in the last run).
* Licences, gated SubPOP access and the manual WVS / Afrobarometer downloads are as in the first notebook.

## 1. Install and log in

In [ ]:
# %%capture
!pip install -q unsloth
!pip install -q --upgrade "transformers>=5" datasets peft pyreadstat huggingface_hub pandas safetensors
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# OPTIONAL: fast kernels for the linear-attention layers (can take several minutes to build; skip if it errors).
# !pip install -q flash-linear-attention causal-conv1d --no-build-isolation

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


In [ ]:
#from huggingface_hub import notebook_login
#notebook_login()          # needed for the gated SubPOP data

import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    try:                                          # Kaggle: Add-ons > Secrets
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:                                          # Colab: the key icon in the left sidebar
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    raise RuntimeError("No Hugging Face token found. Add a secret named HF_TOKEN (Kaggle: Add-ons > Secrets; "
                       "Colab: the key icon), or set the HF_TOKEN environment variable.")

login(token=token)               # needed for the gated SubPOP data
os.environ["HF_TOKEN"] = token   # so every later download in this session sends it too
print("logged in as", __import__("huggingface_hub").whoami()["name"])

## 2. The code, written by cells

Five files in a `survey_ft/` package: `common.py` (example format, prompt, option tokens, splitting), `loaders.py` (one parser per dataset, with the demographic-column check), `build.py` (assemble train / held-out), `train.py` (Unsloth loading with `targets="auto"`, loss, Trainer with checkpoints, evaluation, step timing), `selftest.py`.

In [ ]:
import os
os.makedirs('survey_ft', exist_ok=True)
open('survey_ft/__init__.py', 'w').close()

In [ ]:
%%writefile survey_ft/common.py
"""Shared pieces: the example format, prompt rendering, option-token lookup, splitting."""

import ast
import json
import re
import zlib
from dataclasses import dataclass

import numpy as np

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MAX_OPTIONS = len(LETTERS)


@dataclass
class Example:
    prompt: str      # everything the model reads; ends with "Answer:"
    labels: list     # option labels exactly as the prompt shows them ("A", "B", ... or the source's own)
    target: list     # probabilities over the options, sums to 1
    source: str      # dataset name
    kind: str        # "group": target is a distribution over a subpopulation; "individual": one person's answer
    group: str       # question id (group-level) or respondent id (individual); used to hold groups out


def clean(text, limit=140):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= limit else text[:limit - 1] + "…"


def render_question(question, options):
    lines = ["Question: " + clean(question, 400), "Options:"]
    lines += ["%s. %s" % (LETTERS[i], clean(o)) for i, o in enumerate(options)]
    lines += ["Answer with the letter of the option.", "Answer:"]
    return "\n".join(lines)


def build_prompt(context, question, options):
    head = (context.strip() + "\n\n") if context and context.strip() else ""
    return head + render_question(question, options)


def normalise(p):
    """A proper probability vector, or None if the input cannot be made into one."""
    try:
        p = np.asarray(p, dtype=float)
    except (TypeError, ValueError):
        return None
    if p.ndim != 1 or len(p) < 2 or not np.isfinite(p).all():
        return None
    p = np.clip(p, 0.0, None)
    total = p.sum()
    return None if total <= 0 else (p / total).tolist()


def as_obj(value):
    """A dict/list that may have arrived as a JSON string or a Python repr."""
    if isinstance(value, (dict, list)):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except ValueError:
        pass
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None


def label_token_ids(tok, labels):
    """One token id per option label (the token the model would emit after "Answer:"),
    or None if some label is not a single distinct token (the example is then unusable)."""
    ids = []
    for label in labels:
        pieces = tok.encode(" " + str(label), add_special_tokens=False)
        while pieces and not tok.decode([pieces[0]]).strip():     # a bare-space token in front
            pieces = pieces[1:]
        if len(pieces) != 1:
            return None
        ids.append(pieces[0])
    return ids if len(set(ids)) == len(ids) else None


def hold_out(group, fraction, salt="v1"):
    """Deterministic: True for roughly `fraction` of group ids, always the same ones."""
    return (zlib.crc32((salt + str(group)).encode()) % 10000) / 10000.0 < fraction


def split_by_group(examples, eval_fraction, salt="v1"):
    train, held = [], []
    for ex in examples:
        (held if hold_out(ex.source + "|" + ex.group, eval_fraction, salt) else train).append(ex)
    return train, held

In [ ]:
%%writefile survey_ft/loaders.py
"""One loader per dataset. Each has a `fetch_*` (network or file access) and a pure `parse_*`
that turns rows into Examples; the parsers are what the self-test exercises."""

import ast
import random
import os
import re

import numpy as np

from .common import (Example, as_obj, build_prompt, clean, normalise, MAX_OPTIONS, LETTERS)


# ---------------------------------------------------------------- group-level --
class _Safe(dict):
    def __missing__(self, key):
        return "{" + key + "}"


def _fill(template, mapping):
    """str.format with a mapping, leaving unknown {placeholders} untouched."""
    try:
        return str(template).format_map(_Safe(mapping if isinstance(mapping, dict) else {}))
    except (ValueError, IndexError, KeyError, AttributeError):
        return str(template)


def _option_texts(answer_options, labels):
    """{label: text} from SimBench's `answer_options` (a dict, or a list aligned with the labels), else {}."""
    opts = as_obj(answer_options)
    if isinstance(opts, dict):
        return {str(k): str(v) for k, v in opts.items()}
    if isinstance(opts, list) and len(opts) == len(labels):
        return {l: str(o) for l, o in zip(labels, opts)}
    return {}


def parse_simbench(rows, limit=None):
    """SimBench rows. Columns used: group_prompt_template + group_prompt_variable_map (the persona), input_template
    (the question; in SimBenchGrouped it may have {placeholders} filled from input_variable_map), answer_options
    (appended as lettered lines if the question text does not already show them) and human_answer
    {option label: proportion}. SimBenchPop and SimBenchGrouped have different columns, so rows may lack some."""
    out = []
    for r in rows:
        answer = as_obj(r.get("human_answer"))
        if not isinstance(answer, dict) or len(answer) < 2:
            continue
        labels = [str(k) for k in answer]
        target = normalise([answer[k] for k in answer])
        if target is None:
            continue
        template = str(r.get("group_prompt_template") or "")
        persona = _fill(template, as_obj(r.get("group_prompt_variable_map")) or {})
        stem = _fill(r.get("input_template") or "", as_obj(r.get("input_variable_map")) or {}).strip()
        if not stem or stem == "nan":
            continue
        texts = _option_texts(r.get("answer_options"), labels)
        if texts and sum(t in stem for t in texts.values()) < max(1, len(texts) // 2):
            stem += "\nOptions:\n" + "\n".join("%s. %s" % (k, clean(v)) for k, v in texts.items())
        prompt = (persona.strip() + "\n\n" if persona.strip() else "") + stem + "\nAnswer:"
        out.append(Example(prompt, labels, target, "SimBench:" + str(r.get("dataset_name", "")),
                           "group", stem[:200]))
        if limit and len(out) >= limit:
            break
    return out


def fetch_simbench(files=("SimBenchPop.csv", "SimBenchGrouped.csv")):
    """Read each CSV on its own: they have different columns, so datasets.load_dataset cannot merge them."""
    import pandas as pd
    from huggingface_hub import hf_hub_download
    rows = []
    for name in files:
        df = pd.read_csv(hf_hub_download("pitehu/SimBench", name, repo_type="dataset"))
        df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])
        rows.extend(df.astype(object).where(df.notna(), None).to_dict("records"))
    return rows

def parse_subpop(rows, limit=None):
    """SubPOP rows: qkey, attribute, group, question, options, responses, refusal_rate."""
    out = []
    for r in rows:
        options, responses = r.get("options"), r.get("responses")
        if (isinstance(options, list) and isinstance(responses, list) and len(options) == len(responses) + 1
                and any(w in str(options[-1]).lower() for w in ("refus", "no answer"))):
            options = options[:-1]          # the refusal share is the separate refusal_rate field, not in responses
        if not options or not responses or len(options) != len(responses) or len(options) > MAX_OPTIONS:
            continue
        target = normalise(responses)
        if target is None:
            continue
        context = "Survey respondents: U.S. adults. Subpopulation (%s): %s." % (
            clean(r.get("attribute", "")), clean(r.get("group", "")))
        out.append(Example(build_prompt(context, r.get("question", ""), options),
                           list(LETTERS[:len(options)]), target, "SubPOP", "group",
                           str(r.get("qkey", r.get("question", ""))[:200])))
        if limit and len(out) >= limit:
            break
    return out


def fetch_subpop():
    """Gated on Hugging Face: log in and accept the terms on the dataset page first."""
    import json
    from huggingface_hub import hf_hub_download
    out = {}
    for name in ("subpop_train.jsonl", "subpop_eval.jsonl"):
        path = hf_hub_download("jjssuh/subpop", name, repo_type="dataset")
        with open(path, encoding="utf-8") as fh:
            out[name] = [json.loads(line) for line in fh if line.strip()]
    return out["subpop_train.jsonl"], out["subpop_eval.jsonl"]


_DEFAULTDICT = re.compile(r"^\s*defaultdict\(\s*<class 'list'>\s*,\s*(.*)\)\s*$", re.S)


def parse_selections(text):
    """GlobalOpinionQA stores `selections` as the repr of a defaultdict(list)."""
    if isinstance(text, dict):
        return text
    m = _DEFAULTDICT.match(str(text))
    try:
        return ast.literal_eval(m.group(1) if m else str(text))
    except (ValueError, SyntaxError):
        return None


def parse_globalopinionqa(rows, limit=None):
    out = []
    for r in rows:
        options = as_obj(r.get("options"))
        selections = parse_selections(r.get("selections"))
        if not options or not isinstance(selections, dict) or len(options) > MAX_OPTIONS:
            continue
        for country, proportions in selections.items():
            if len(proportions) != len(options):
                continue
            target = normalise(proportions)
            if target is None:
                continue
            context = "Survey respondents: adults in %s." % clean(country)
            out.append(Example(build_prompt(context, r.get("question", ""), options),
                               list(LETTERS[:len(options)]), target, "GlobalOpinionQA", "group",
                               str(r.get("question", ""))[:200]))
        if limit and len(out) >= limit:
            break
    return out


def fetch_globalopinionqa():
    from datasets import load_dataset
    return [dict(r) for r in load_dataset("Anthropic/llm_global_opinions", split="train")]


# ----------------------------------------------------------------- individual --
class Person:
    """One respondent: demographic lines, plus (question id, answer index) pairs
    whose question text and options live in a shared table."""
    __slots__ = ("pid", "demo", "qa")

    def __init__(self, pid, demo, qa):
        self.pid, self.demo, self.qa = pid, demo, qa


def person_examples(people, questions, source, desc, n, seed=0, k_context=(0, 8), max_context_chars=1200):
    """GIVEN -> PREDICT examples: demographics plus a few other answers as context, one more question as target."""
    rng = random.Random(seed)
    people = [p for p in people if len(p.qa) >= 2]
    out = []
    if not people:
        return out
    for _ in range(n):
        p = rng.choice(people)
        t = rng.randrange(len(p.qa))
        qid, answer = p.qa[t]
        qtext, options = questions[qid]
        others = [qa for i, qa in enumerate(p.qa) if i != t]
        rng.shuffle(others)
        lines = list(p.demo)
        for oq, oa in others[:rng.randint(*k_context)]:
            q, opts = questions[oq]
            lines.append("%s: %s" % (clean(q, 90), clean(opts[oa], 60)))
        context, size = [desc, "Respondent:"], len(desc)
        for line in lines:
            size += len(line) + 3
            if size > max_context_chars:
                break
            context.append("- " + line)
        target = [0.0] * len(options)
        target[answer] = 1.0
        out.append(Example(build_prompt("\n".join(context), qtext, options),
                           list(LETTERS[:len(options)]), target, source, "individual", p.pid))
    return out


def _walk_questions(node, block=""):
    """Every dict that looks like a Qualtrics-style question, with the block name it sits in."""
    if isinstance(node, dict):
        block = str(node.get("BlockName", block))
        if "QuestionText" in node and "Options" in node:
            yield block, node
        for value in node.values():
            yield from _walk_questions(value, block)
    elif isinstance(node, list):
        for value in node:
            yield from _walk_questions(value, block)


def parse_twin(records):
    """records: (pid, persona_json) pairs, persona_json a dict/list or its JSON string.
    Single-choice questions only; the block called "demographics" becomes the profile."""
    people, questions, qindex = [], {}, {}
    for pid, raw in records:
        obj = as_obj(raw)
        if obj is None:
            continue
        demo, qa = [], []
        for block, q in _walk_questions(obj):
            options = q.get("Options")
            if isinstance(options, dict):
                options = list(options.values())
            if not isinstance(options, list) or not (2 <= len(options) <= MAX_OPTIONS):
                continue
            options = [o["text"] if isinstance(o, dict) and "text" in o else o for o in options]
            if not all(isinstance(o, str) for o in options):
                continue
            selected = (q.get("Answers") or {}).get("SelectedText")
            if isinstance(selected, list) and len(selected) == 1:
                selected = selected[0]
            if not isinstance(selected, str) or selected not in options:
                continue
            text = re.sub(r"<[^>]+>", " ", str(q.get("QuestionText", "")))
            if "demograph" in block.lower():
                demo.append("%s: %s" % (clean(text, 90), clean(selected, 60)))
                continue
            key = (text, tuple(options))
            if key not in qindex:
                qindex[key] = "twin-%d" % len(qindex)
                questions[qindex[key]] = (text, options)
            qa.append((qindex[key], options.index(selected)))
        people.append(Person(str(pid), demo, qa))
    return people, questions


def fetch_twin():
    from datasets import load_dataset
    data = load_dataset("LLM-Digital-Twin/Twin-2K-500", "wave_split")["data"]
    return [(r["pid"], r["wave1_3_persona_json"]) for r in data]


MISSING = re.compile(r"missing|not asked|not applicable|inapplicable|no answer|not available|n\.?a\.?\b|"
                     r"not coded|dropped|multiple answers|inapplicable", re.I)
EXCLUDE_VARS = re.compile(r"(^|_)(id|weight|wt|wgt|date|time|stratum|psu|interviewer|int_|version|lang)\b|"
                          r"weight|interview|serial|respno|resp_no|^s0|^s1|^s2|^s3", re.I)

# Each demographic role: (role, explicit variable names, regex on the variable label, regex that at least one VALUE label
# must match). A candidate column is accepted only if its value labels look like the role (a "sex" column has to
# offer male/female), which is what stops a label search from picking e.g. a "justifiable: sex before marriage" item.
WVS_CFG = dict(
    name="WVS", desc="World Values Survey: a multi-country survey of people's values and beliefs.",
    demo=[("country", ["B_COUNTRY_ALPHA", "B_COUNTRY", "COUNTRY_ALPHA"], r"\bcountry\b", None),
          ("sex", ["Q260", "X001"], r"^(sex|gender)\b|respondent.?s? (sex|gender)", r"\b(male|female|man|woman)\b"),
          ("age", ["Q262", "X003"], r"^age\b|\bage\b", None),
          ("education", ["Q275", "Q275R", "X025R", "X025"], r"education", r"(educat|school|universit|college|primary|secondary|degree)"),
          ("marital status", ["Q273", "X007"], r"marital", r"(married|divorced|widow|single|separated)"),
          ("employment", ["Q279", "X028"], r"employment", r"((full|part)[- ]?time|retired|unemployed|housewife|student|self[- ]?employed)")],
    numeric={"age"})
AFROBAROMETER_CFG = dict(
    name="Afrobarometer", desc="Afrobarometer: a survey of public attitudes in African countries.",
    demo=[("country", ["COUNTRY"], r"\bcountry\b", None),
          ("urban or rural", ["URBRUR"], r"urban|rural", r"(urban|rural)"),
          ("age", ["Q1"], r"age of respondent|^age\b|respondent.?s? age", None),
          ("gender", ["Q100", "Q101"], r"(gender|sex) of respondent|respondent.?s? (gender|sex)|^gender", r"\b(male|female|man|woman)\b"),
          ("education", ["Q94"], r"education", r"(educat|school|universit|college|primary|secondary|degree)"),
          ("religion", ["Q95", "Q98"], r"religio", r"(christian|muslim|catholic|protestant|islam|traditional|no religion|hindu|jewish|orthodox)")],
    numeric={"age"})


_READ_CACHE = {}


def _read_labelled(path):
    """(DataFrame, metadata) of an SPSS/Stata file, kept in memory so the audit and both loads read it once."""
    key = (str(path), os.path.getmtime(path))
    if key not in _READ_CACHE:
        import pyreadstat
        reader = pyreadstat.read_dta if str(path).lower().endswith(".dta") else pyreadstat.read_sav
        _READ_CACHE.clear()
        _READ_CACHE[key] = reader(path, apply_value_formats=False)
    return _READ_CACHE[key]


def _value_labels_match(value_labels, column, pattern):
    return column in value_labels and any(re.search(pattern, str(l), re.I) for l in value_labels[column].values())


def demographic_candidates(df, meta, role_spec):
    """Columns that could play this role: the explicit names first, then any column whose label matches."""
    role, names, label_pattern, _ = role_spec
    var_label = dict(zip(meta.column_names, meta.column_labels))
    cols = list(df.columns)
    return [c for c in names if c in cols] + [c for c in cols if c not in names and
                                              re.search(label_pattern, str(var_label.get(c) or ""), re.I)]


def choose_demographics(df, meta, cfg, override=None):
    """{role: column}. A candidate is accepted only if its values look like the role (see the config comment)."""
    value_labels = meta.variable_value_labels
    chosen = {}
    for spec in cfg["demo"]:
        role, names, label_pattern, value_pattern = spec
        if override and role in override:
            if override[role] in df.columns:
                chosen[role] = override[role]
            continue
        for c in demographic_candidates(df, meta, spec):
            if role in cfg.get("numeric", ()):
                v = np.asarray(df[c], dtype=float) if np.issubdtype(df[c].dtype, np.number) else np.array([])
                v = v[np.isfinite(v)]
                if len(v) and 10 <= np.median(v) <= 80 and np.mean((v > 0) & (v < 120)) > 0.9:
                    chosen[role] = c
                    break
            elif value_pattern is None or _value_labels_match(value_labels, c, value_pattern):
                chosen[role] = c
                break
    return chosen


def audit_demographics(path, cfg, max_rows=5000, demo_override=None):
    """Print, for every role, the column chosen, its label and its commonest values, and the other candidates.
    Read this once per dataset: a sex column whose values are 'Never justifiable' is a wrong column."""
    import pandas as pd
    df, meta = _read_labelled(path)
    df = df.head(max_rows)
    var_label = dict(zip(meta.column_names, meta.column_labels))
    value_labels = meta.variable_value_labels
    chosen = choose_demographics(df, meta, cfg, demo_override)
    rows = []
    for spec in cfg["demo"]:
        role = spec[0]
        c = chosen.get(role)
        if c is None:
            print("%-16s NOT FOUND   (candidates: %s)" % (role, demographic_candidates(df, meta, spec)[:6]))
            continue
        counts = df[c].value_counts(dropna=True).head(4)
        shown = [("%s (%d)" % (value_labels.get(c, {}).get(k, k), n)) for k, n in counts.items()]
        print("%-16s %-14s %-48s %s" % (role, c, str(var_label.get(c))[:48], "; ".join(str(x)[:30] for x in shown)))
        others = [x for x in demographic_candidates(df, meta, spec) if x != c][:4]
        if others:
            print("%-16s   other candidates: %s" % ("", ", ".join("%s [%s]" % (x, str(var_label.get(x))[:30]) for x in others)))
    return chosen


def load_survey_people(path, cfg, max_people=20000, seed=0, max_levels=MAX_OPTIONS, demo_override=None):
    """SPSS/Stata survey file -> (people, questions). Question text is the variable label, options are
    the value labels in code order. "Don't know" and "refused" stay as ordinary options; labels that
    mean the question was not asked or is missing drop that respondent-question pair."""
    df, meta = _read_labelled(path)
    if len(df) > max_people:
        df = df.sample(max_people, random_state=seed).reset_index(drop=True)
    var_label = dict(zip(meta.column_names, meta.column_labels))
    value_labels = meta.variable_value_labels
    cols = list(df.columns)

    demo_cols = choose_demographics(df, meta, cfg, demo_override)
    print("%s: demographic variables used -> %s" % (cfg["name"], demo_cols))

    questions, qcols = {}, {}
    for c in cols:
        if c in demo_cols.values() or c not in value_labels or EXCLUDE_VARS.search(c):
            continue
        valid = {code: str(lab) for code, lab in value_labels[c].items() if not MISSING.search(str(lab))}
        if not (2 <= len(valid) <= max_levels):
            continue
        codes = sorted(valid, key=lambda k: (bool(re.search(r"don.?t know|refus|no opinion", valid[k], re.I)), k))
        qid = "%s-%s" % (cfg["name"], c)
        questions[qid] = (str(var_label.get(c) or c), [valid[k] for k in codes])
        qcols[c] = (qid, {code: i for i, code in enumerate(codes)})
    print("%s: %d usable question variables" % (cfg["name"], len(qcols)))

    index = {c: df[c].map(m[1]).to_numpy(dtype=float, na_value=np.nan) for c, m in qcols.items()}
    people = []
    for i in range(len(df)):
        demo = []
        for role, c in demo_cols.items():
            v = df.at[i, c]
            if v is None or (isinstance(v, float) and np.isnan(v)):
                continue
            if role in cfg.get("numeric", ()):
                if 0 < v < 120:
                    demo.append("%s: %d" % (role, int(v)))
            elif c in value_labels:
                lab = value_labels[c].get(v)
                if lab is not None and not MISSING.search(str(lab)):
                    demo.append("%s: %s" % (role, clean(lab, 60)))
            elif isinstance(v, str) and v.strip():
                demo.append("%s: %s" % (role, clean(v, 60)))
        qa = [(qcols[c][0], int(index[c][i])) for c in qcols if not np.isnan(index[c][i])]
        people.append(Person("%s-%s-%d" % (cfg["name"], seed, i), demo, qa))
    return people, questions

In [ ]:
%%writefile survey_ft/build.py
"""Turn the per-dataset example lists into one shuffled training list and one held-out list."""

import random
from collections import defaultdict

from .common import split_by_group


def family(example):
    return example.source.split(":")[0]


def assemble(sets, official_eval=None, eval_fraction=0.05, max_train_per_dataset=40000,
             eval_cap_per_source=300, seed=0):
    """sets: {dataset: [Example]}. Datasets with an official held-out file (SubPOP) use it; the others hold out
    whole questions (group-level) or whole respondents (individual-level), so nothing leaks through a shared question or person."""
    rng = random.Random(seed)
    official_eval = official_eval or {}
    train, held = [], []
    for name, examples in sets.items():
        if name in official_eval:
            tr, ev = examples, official_eval[name]
        else:
            tr, ev = split_by_group(examples, eval_fraction)
        if len(tr) > max_train_per_dataset:
            tr = rng.sample(tr, max_train_per_dataset)
        train += tr
        held += ev
    by_source = defaultdict(list)
    for ex in held:
        by_source[ex.source].append(ex)
    held = [ex for exs in by_source.values()
            for ex in (rng.sample(exs, eval_cap_per_source) if len(exs) > eval_cap_per_source else exs)]
    rng.shuffle(train)
    return train, held


def summary(train, held):
    import pandas as pd
    rows = defaultdict(lambda: {"train": 0, "held_out": 0})
    for ex in train:
        rows[ex.source]["train"] += 1
    for ex in held:
        rows[ex.source]["held_out"] += 1
    return pd.DataFrame(rows).T.sort_index()

In [ ]:
%%writefile survey_ft/train.py
"""Model loading (Unsloth), the option-letter loss, a Trainer around it, and evaluation."""

import re
import time

import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

from .common import label_token_ids

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def text_tokenizer(tok):
    """Qwen3.5 is a vision-language model, so Unsloth may hand back a processor; we only need its text tokenizer."""
    tok = getattr(tok, "tokenizer", tok)
    tok.padding_side = "left"          # the next-token logits we read are at the last position
    tok.truncation_side = "left"       # if a prompt is too long, lose the start, never the question
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def discover_targets(model, linear_attention=True, skip=("visual", "vision", "lm_head", "embed")):
    """LoRA target suffixes for every nn.Linear inside the language layers, found from the model itself.
    Qwen3.5 is hybrid: 3 of 4 layers are linear attention (in_proj_qkv, in_proj_z, in_proj_b, in_proj_a, out_proj),
    which the usual q/k/v/o + MLP list never touches. linear_attention=False drops them again."""
    suffixes = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear) or "layers." not in name or any(w in name for w in skip):
            continue
        if not linear_attention and "linear_attn" in name:
            continue
        last = name.split(".")[-1]
        if last not in suffixes:
            suffixes.append(last)
    return suffixes or list(LORA_TARGETS)


def describe_adapters(model):
    """How many LoRA-wrapped modules there are per module type, and the trainable parameter count."""
    kinds = {}
    for name, p in model.named_parameters():
        if "lora_A" in name:
            kind = re.sub(r"\.lora_A.*", "", name).split("layers.")[-1]
            kind = re.sub(r"^\d+\.", "", kind)
            kinds[kind] = kinds.get(kind, 0) + 1
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return kinds, trainable


def load_model(model_name, max_seq_len=1024, backend="unsloth", lora_r=16, lora_alpha=16,
               targets=LORA_TARGETS, dtype=None):
    """targets: a list of module suffixes, or "auto" (every Linear in the language layers, including Qwen3.5's
    linear-attention projections), or "no_linear_attn" (auto minus those: the original coverage's equivalent)."""
    if backend == "unsloth":
        from unsloth import FastLanguageModel
        model, tok = FastLanguageModel.from_pretrained(
            model_name=model_name, max_seq_length=max_seq_len,
            load_in_4bit=False, load_in_16bit=True, full_finetuning=False)    # Unsloth: no QLoRA for Qwen3.5
        if isinstance(targets, str):
            targets = discover_targets(model, linear_attention=(targets == "auto"))
        print("LoRA targets:", targets)
        model = FastLanguageModel.get_peft_model(
            model, r=lora_r, target_modules=targets, lora_alpha=lora_alpha, lora_dropout=0,
            bias="none", use_gradient_checkpointing="unsloth", random_state=3407,
            max_seq_length=max_seq_len)
        kinds, trainable = describe_adapters(model)
        print("adapters per module type:", kinds, "| trainable parameters: %d" % trainable)
        return model, text_tokenizer(tok)
    from peft import LoraConfig, get_peft_model        # plain transformers + peft, used by the CPU self-test
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
    if isinstance(targets, str):
        targets = discover_targets(model, linear_attention=(targets == "auto"))
    model = get_peft_model(model, LoraConfig(r=lora_r, lora_alpha=lora_alpha, target_modules=targets,
                                             lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"))
    kinds, trainable = describe_adapters(model)
    print("LoRA targets:", targets, "| adapters per module type:", kinds, "| trainable parameters: %d" % trainable)
    return model, text_tokenizer(tok)


def prepare(examples, tok, weights=None):
    """Drop examples whose option labels are not single distinct tokens; attach token ids and weights."""
    weights = weights or {}
    kept, dropped = [], 0
    cache = {}
    for ex in examples:
        key = tuple(ex.labels)
        if key not in cache:
            cache[key] = label_token_ids(tok, ex.labels)
        ids = cache[key]
        if ids is None or len(ids) != len(ex.target):
            dropped += 1
            continue
        kept.append({"prompt": ex.prompt, "label_ids": ids, "target": ex.target,
                     "weight": float(weights.get(ex.source.split(":")[0], 1.0)),
                     "source": ex.source, "kind": ex.kind})
    print("prepared %d examples, dropped %d (labels not single tokens)" % (len(kept), dropped))
    return kept


class ListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


class Collator:
    def __init__(self, tok, max_len):
        self.tok, self.max_len = tok, max_len

    def __call__(self, rows):
        enc = self.tok([r["prompt"] for r in rows], return_tensors="pt", padding=True,
                       truncation=True, max_length=self.max_len)
        width = max(len(r["label_ids"]) for r in rows)
        ids = torch.zeros(len(rows), width, dtype=torch.long)
        mask = torch.zeros(len(rows), width, dtype=torch.bool)
        target = torch.zeros(len(rows), width)
        for i, r in enumerate(rows):
            k = len(r["label_ids"])
            ids[i, :k] = torch.tensor(r["label_ids"])
            mask[i, :k] = True
            target[i, :k] = torch.tensor(r["target"])
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
                "label_ids": ids, "label_mask": mask, "target": target,
                "weight": torch.tensor([r["weight"] for r in rows])}


_KEEP_ONE = {"ok": True}


def last_logits(model, input_ids, attention_mask):
    """Next-token logits at the final position. `logits_to_keep=1` avoids materialising the full
    sequence x vocabulary logits; it is skipped if this model's forward does not accept it."""
    if _KEEP_ONE["ok"]:
        try:
            return model(input_ids=input_ids, attention_mask=attention_mask, logits_to_keep=1).logits[:, -1, :]
        except (TypeError, ValueError):
            _KEEP_ONE["ok"] = False
    return model(input_ids=input_ids, attention_mask=attention_mask).logits[:, -1, :]


def option_logprobs(logits, label_ids, label_mask):
    """log-softmax restricted to the option tokens: what the model says about the answer, normalised over options."""
    chosen = logits.float().gather(1, label_ids).masked_fill(~label_mask, torch.finfo(torch.float32).min)
    return torch.log_softmax(chosen, dim=1)


def survey_loss(logits, label_ids, label_mask, target, weight):
    """Cross-entropy of the target distribution under the model's option distribution. For a group-level
    target this is the forward KL up to a constant (the target's entropy); for a one-hot answer it is the log loss."""
    logp = option_logprobs(logits, label_ids, label_mask)
    per_example = -(target * logp.masked_fill(~label_mask, 0.0)).sum(1)
    return (per_example * weight).sum() / weight.sum().clamp(min=1e-8)


class SurveyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        logits = last_logits(model, inputs["input_ids"], inputs["attention_mask"])
        loss = survey_loss(logits, inputs["label_ids"], inputs["label_mask"], inputs["target"], inputs["weight"])
        return (loss, {"logits": logits}) if return_outputs else loss


def make_args(output_dir="outputs", epochs=1.0, lr=2e-4, batch_size=8, grad_accum=4, max_steps=-1,
              logging_steps=10, warmup_ratio=0.03, save_steps=None):
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.is_bf16_supported()
    kw = dict(output_dir=output_dir, num_train_epochs=epochs, max_steps=max_steps, learning_rate=lr,
              per_device_train_batch_size=batch_size, gradient_accumulation_steps=grad_accum,
              lr_scheduler_type="cosine", logging_steps=logging_steps,
              optim="adamw_8bit" if cuda else "adamw_torch", weight_decay=0.0,
              bf16=bf16, fp16=cuda and not bf16, report_to="none",
              save_strategy="steps" if save_steps else "no", save_total_limit=2,
              **({"save_steps": int(save_steps)} if save_steps else {}),
              remove_unused_columns=False, dataloader_num_workers=0, seed=3407)
    try:
        return TrainingArguments(warmup_ratio=warmup_ratio, **kw)      # transformers 4.x
    except TypeError:
        return TrainingArguments(warmup_steps=warmup_ratio, **kw)      # transformers 5: a float below 1 is a ratio


def train(model, tok, rows, max_len=1024, resume=None, **kwargs):
    """resume: a checkpoint folder under output_dir (see save_steps), or True for the latest one."""
    trainer = SurveyTrainer(model=model, args=make_args(**kwargs), train_dataset=ListDataset(rows),
                            data_collator=Collator(tok, max_len))
    return trainer.train(resume_from_checkpoint=resume)


@torch.no_grad()
def evaluate(model, tok, rows, batch_size=16, max_len=1024):
    """Per-source mean of: kl (target || model), w1 (ordinal Wasserstein-1, scaled to [0,1]),
    top1 (does the model's favourite option match the target's), logscore (E_target log p_model),
    plus uniform_kl, the same KL for a model that spreads probability evenly."""
    model.eval()
    collate, device = Collator(tok, max_len), next(model.parameters()).device
    stats = {}
    for s in range(0, len(rows), batch_size):
        chunk = rows[s:s + batch_size]
        batch = {k: v.to(device) for k, v in collate(chunk).items()}
        logp = option_logprobs(last_logits(model, batch["input_ids"], batch["attention_mask"]),
                               batch["label_ids"], batch["label_mask"]).cpu().numpy()
        for r, lp in zip(chunk, logp):
            k = len(r["target"])
            lp, t = lp[:k], np.asarray(r["target"])
            p = np.exp(lp)
            kl = float(np.sum(t[t > 0] * (np.log(t[t > 0]) - lp[t > 0])))
            w1 = float(np.abs(np.cumsum(p) - np.cumsum(t)).sum() / max(k - 1, 1))
            uni = float(np.sum(t[t > 0] * (np.log(t[t > 0]) + np.log(k))))
            stats.setdefault(r["source"], []).append(
                (kl, w1, float(np.argmax(p) == np.argmax(t)), float((t * lp).sum()), uni))
    out = {}
    for source, vals in sorted(stats.items()):
        m = np.mean(vals, axis=0)
        out[source] = dict(n=len(vals), kl=m[0], w1=m[1], top1=m[2], logscore=m[3], uniform_kl=m[4])
    return out


def seconds_per_step(model, tok, rows, batch_size=8, grad_accum=4, max_len=1024, probes=2):
    """Time `probes` forward+backward micro-batches WITHOUT an optimiser step (the adapter is left unchanged), and
    scale to one optimiser step. Use it to turn a time budget into a step count before training."""
    model.train()
    collate, device = Collator(tok, max_len), next(model.parameters()).device
    rows = rows[:batch_size * probes]
    times = []
    for s in range(0, len(rows), batch_size):
        batch = {k: v.to(device) for k, v in collate(rows[s:s + batch_size]).items()}
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        loss = survey_loss(last_logits(model, batch["input_ids"], batch["attention_mask"]),
                           batch["label_ids"], batch["label_mask"], batch["target"], batch["weight"])
        loss.backward()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - t0)
        model.zero_grad(set_to_none=True)
    # the first micro-batch carries warm-up cost; use the fastest as the steady-state figure
    return min(times) * grad_accum


def save_adapter(model, tok, path):
    model.save_pretrained(path)
    tok.save_pretrained(path)
    return path

In [ ]:
%%writefile survey_ft/selftest.py
"""CPU self-test: fake data in each dataset's format, the real parsers, the real loss and Trainer, and a tiny
random model with a toy tokenizer. It checks the plumbing and that the loss can fall on a planted signal.
It says nothing about the real datasets or the real model."""

import os
import random
import tempfile

import numpy as np
import pandas as pd
import torch

from . import loaders, train
from .build import assemble, summary


def fixtures(seed=0):
    rnd = random.Random(seed)
    simbench, subpop, gopqa, twin = [], [], [], []
    for i in range(30):
        for age, base in (("young", [0.7, 0.2, 0.1]), ("old", [0.1, 0.2, 0.7])):
            simbench.append({
                "dataset_name": "OpinionQA", "group_prompt_template": "You are a {age_group} respondent from {country}.",
                "group_prompt_variable_map": str({"age_group": age, "country": "Nigeria"}),
                "input_template": "How satisfied are you with item %d?\nA. Very\nB. Somewhat\nC. Not at all" % i,
                "human_answer": str({"A": base[0], "B": base[1], "C": base[2]}), "group_size": 100, "auxiliary": {}})
    for i in range(30):
        for party, base in (("Democrat", [0.8, 0.15, 0.05]), ("Republican", [0.1, 0.8, 0.1])):
            subpop.append({"qkey": "Q%d" % i, "attribute": "POLPARTY", "group": party,
                           "question": "Do you support policy %d?" % i, "options": ["Yes", "No", "Refused"],
                           "responses": base, "refusal_rate": 0.05})
    subpop.append({"qkey": "bad", "attribute": "X", "group": "Y", "question": "bad", "options": ["Yes", "No"],
                   "responses": [1.0], "refusal_rate": 0.0})
    for i in range(30):
        gopqa.append({"question": "How much do you trust institution %d?" % i, "source": "WVS",
                      "options": str(["A lot", "A little", "Not at all"]),
                      "selections": "defaultdict(<class 'list'>, {'Nigeria': [0.7, 0.2, 0.1], 'Kenya': [0.1, 0.3, 0.6]})"})
    for pid in range(80):
        gender = rnd.choice(["Male", "Female"])
        agree = "Agree" if gender == "Male" else "Disagree"
        questions = [{"QuestionText": "What is your gender?", "QuestionType": "MC", "Options": ["Male", "Female"],
                      "Answers": {"SelectedText": gender}}]
        block2 = [{"QuestionText": "<b>Statement %d</b>" % j, "QuestionType": "MC", "Options": ["Agree", "Disagree"],
                   "Answers": {"SelectedText": agree if rnd.random() < 0.9 else rnd.choice(["Agree", "Disagree"])}}
                  for j in range(8)]
        twin.append((pid, [{"BlockName": "Demographics", "Questions": questions},
                           {"BlockName": "Opinions", "Questions": block2}]))
    return simbench, subpop, gopqa, twin


def spss_fixtures(folder, seed=0):
    rng = np.random.RandomState(seed)
    n = 600
    group = rng.randint(1, 3, n)                       # 1/2: drives the answers
    def answers(strength, levels):
        out = np.where(rng.rand(n) < strength, group, rng.randint(1, levels + 1, n))
        return np.where(rng.rand(n) < 0.05, -4, out)   # some "not asked"
    wvs = pd.DataFrame({"B_COUNTRY_ALPHA": rng.choice(["NGA", "KEN"], n), "X001": group,
                        "X003": rng.randint(18, 80, n), "W_WEIGHT": rng.rand(n)})
    wlabels, wvals = {"B_COUNTRY_ALPHA": "Country code", "X001": "Sex", "X003": "Age", "W_WEIGHT": "Weight"}, {}
    wvals["X001"] = {1: "Male", 2: "Female"}
    for j in range(6):
        wvs["Q%d" % (j + 1)] = answers(0.8, 3)
        wlabels["Q%d" % (j + 1)] = "Important in life: topic %d" % (j + 1)
        wvals["Q%d" % (j + 1)] = {1: "Very important", 2: "Rather important", 3: "Not important",
                                  -1: "Don't know", -4: "Not asked", -5: "Missing; Not available"}
    import pyreadstat
    p1 = os.path.join(folder, "wvs.sav")
    pyreadstat.write_sav(wvs, p1, column_labels=[wlabels[c] for c in wvs.columns], variable_value_labels=wvals)

    ab = pd.DataFrame({"COUNTRY": rng.randint(1, 3, n), "URBRUR": rng.randint(1, 3, n), "Q1": rng.randint(18, 90, n),
                       "Q101": group})
    alabels = {"COUNTRY": "Country", "URBRUR": "Urban or Rural Primary Sampling Unit", "Q1": "Age of respondent",
               "Q101": "Gender of respondent"}
    avals = {"COUNTRY": {1: "Nigeria", 2: "Kenya"}, "URBRUR": {1: "Urban", 2: "Rural"}, "Q101": {1: "Male", 2: "Female"}}
    for j, name in enumerate(["Q22A", "Q22B", "Q22C", "Q22D", "Q22E"]):
        ab[name] = np.where(rng.rand(n) < 0.05, 9, answers(0.8, 3))
        alabels[name] = "Would you approve of option %d?" % j
        avals[name] = {1: "Disapprove", 2: "Neutral", 3: "Approve", 9: "Don't know", -1: "Missing"}
    p2 = os.path.join(folder, "afrobarometer.sav")
    pyreadstat.write_sav(ab, p2, column_labels=[alabels[c] for c in ab.columns], variable_value_labels=avals)
    return p1, p2


def toy_tokenizer(texts):
    from tokenizers import Tokenizer, models, pre_tokenizers
    from transformers import PreTrainedTokenizerFast
    words = {"[PAD]", "[UNK]", "[EOS]"}
    pre = pre_tokenizers.Whitespace()
    for t in texts:
        words.update(w for w, _ in pre.pre_tokenize_str(t))
    words.update("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
    vocab = {w: i for i, w in enumerate(sorted(words))}
    tk = Tokenizer(models.WordLevel(vocab, unk_token="[UNK]"))
    tk.pre_tokenizer = pre
    return PreTrainedTokenizerFast(tokenizer_object=tk, pad_token="[PAD]", unk_token="[UNK]", eos_token="[EOS]")


def tiny_model(vocab_size):
    from transformers import AutoModelForCausalLM, Qwen3Config
    kw = dict(vocab_size=vocab_size, hidden_size=64, intermediate_size=128, num_hidden_layers=2,
              num_attention_heads=4, num_key_value_heads=2, head_dim=16, max_position_embeddings=512)
    torch.manual_seed(0)
    return AutoModelForCausalLM.from_config(Qwen3Config(**kw)), "qwen3 (tiny, random)"


def run(steps=80, verbose=True):
    simbench, subpop, gopqa, twin = fixtures()
    sets = {
        "SimBench": loaders.parse_simbench(simbench),
        "SubPOP": loaders.parse_subpop(subpop),
        "GlobalOpinionQA": loaders.parse_globalopinionqa(gopqa),
    }
    people, qs = loaders.parse_twin(twin)
    sets["Twin-2K-500"] = loaders.person_examples(people, qs, "Twin-2K-500", "A survey of US adults.", 600, seed=1)
    with tempfile.TemporaryDirectory() as tmp:
        wvs_path, ab_path = spss_fixtures(tmp)
        for cfg, path in ((loaders.WVS_CFG, wvs_path), (loaders.AFROBAROMETER_CFG, ab_path)):
            persons, questions = loaders.load_survey_people(path, cfg, max_people=600)
            sets[cfg["name"]] = loaders.person_examples(persons, questions, cfg["name"], cfg["desc"], 600, seed=2)
    for name, exs in sets.items():
        assert exs, "no examples parsed for " + name
        assert all(abs(sum(e.target) - 1) < 1e-6 and len(e.target) == len(e.labels) for e in exs), name
        if verbose:
            print("%-16s %5d examples, e.g.:\n%s\n" % (name, len(exs), exs[0].prompt))
    allx = [e for exs in sets.values() for e in exs]
    official = {"SubPOP": loaders.parse_subpop(subpop[:6])}          # stands in for subpop_eval.jsonl
    tr, ev = assemble(sets, official_eval=official, eval_fraction=0.2)
    if verbose:
        print(summary(tr, ev))
    assert tr and ev and all(e.source == "SubPOP" for e in ev if e.source == "SubPOP")
    tok = toy_tokenizer([e.prompt for e in allx])
    tok = train.text_tokenizer(tok)
    model, label = tiny_model(len(tok))
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=train.LORA_TARGETS, task_type="CAUSAL_LM"))
    rows_tr, rows_ev = train.prepare(tr, tok), train.prepare(ev, tok)
    before = train.evaluate(model, tok, rows_ev, max_len=256)
    train.train(model, tok, rows_tr, max_len=256, max_steps=steps, batch_size=16, grad_accum=1,
                lr=3e-3, logging_steps=max(steps // 4, 1))
    after = train.evaluate(model, tok, rows_ev, max_len=256)
    if verbose:
        print("model:", label)
        print(pd.concat({"before": pd.DataFrame(before).T, "after": pd.DataFrame(after).T}, axis=1).round(3).to_string())
    return before, after

## 3. Self-test (CPU, about a minute)

Fake data in each dataset's format, the real parsers, then a tiny random model trained for a few steps. If it fails, the environment (library versions) is off, not the data.

In [ ]:
from survey_ft import selftest
before, after = selftest.run(steps=200, verbose=False)
import pandas as pd
print(pd.concat({"before": pd.DataFrame(before).T[["n", "kl", "w1", "top1"]],
                 "after": pd.DataFrame(after).T[["kl", "w1", "top1"]]}, axis=1).round(3).to_string())

## 4. Settings

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
MAX_LEN = 768
SEED = 0
BACKEND = "unsloth"                      # "transformers" = plain transformers + peft (no Unsloth kernels)

# --- the LoRA: "auto" = every Linear in the language layers, including Qwen3.5's linear-attention projections
LORA_TARGETS = "auto"                    # or "no_linear_attn" to reproduce the first run's coverage
LORA_R, LORA_ALPHA = 16, 16              # the held-out test did not favour a higher rank at this run length

# --- training length. Steps = TRAIN_BUDGET_S / (measured seconds per step x STEP_TIME_FACTOR), unless MAX_STEPS is set.
TRAIN_BUDGET_S = 3 * 3600                # three hours; set to what your session allows
STEP_TIME_FACTOR = 1.3                   # measured 8.0 s/step turned out as 10.3 s/step in the last run
MAX_STEPS = 3440 #None                         # set an int to override the budget
BATCH_SIZE, GRAD_ACCUM, LR = 8, 4, 2e-4
SAVE_STEPS = 100                         # checkpoint (adapter only) every N steps into CKPT_DIR; None to disable
CKPT_DIR = "outputs_ckpt"
RESUME = False                           # True after a crash: continues from the latest checkpoint in CKPT_DIR

# --- data (as in the first run)
MAX_TRAIN_PER_DATASET = 20000
EVAL_FRACTION = 0.05                     # of questions (group-level) or respondents (individual-level), held out whole
EVAL_CAP_PER_SOURCE = 300
INDIVIDUAL_EXAMPLES = 20000              # examples drawn per individual-level dataset
SOURCE_WEIGHTS = {"SimBench": 1.0, "SubPOP": 1.0, "GlobalOpinionQA": 1.0,
                  "Twin-2K-500": 1.0, "WVS": 1.0, "Afrobarometer": 1.0}

# Manual downloads (SPSS .sav). Edit the paths to where your files are.
WVS_PATH = "/kaggle/input/datasets/faithfulchukwunwogor/data-simulacra/F00010733-WVS_Cross-National_Wave_7_spss_v6_0/WVS_Cross-National_Wave_7_spss_v6_0.sav"
AFROBAROMETER_PATH = "/kaggle/input/datasets/faithfulchukwunwogor/data-simulacra/R9.Merge_39ctry.20Nov23.final_.release_Updated.4Jun25-3.sav"
# If the audit in section 5 shows a role mapped to the wrong column, put the right one here.
DEMO_OVERRIDE = {"WVS": None, "Afrobarometer": {"education": "Q94"}}

## 5. Load the six datasets

**Read the demographic audit first.** For each of WVS and Afrobarometer it prints, per role, the column chosen, its question label and its commonest answers: `sex` must show Male/Female (not a justifiability scale), `education` an education scale, and so on. If a role is wrong, set it in `DEMO_OVERRIDE` and re-run. Every dataset then prints a sample prompt.

In [ ]:
from survey_ft import loaders
for cfg, path, ov in ((loaders.WVS_CFG, WVS_PATH, DEMO_OVERRIDE["WVS"]), (loaders.AFROBAROMETER_CFG, AFROBAROMETER_PATH, DEMO_OVERRIDE["Afrobarometer"])):
    print("=" * 30, cfg["name"])
    try:
        loaders.audit_demographics(path, cfg, demo_override=ov)
    except Exception as exc:
        print("skipped:", type(exc).__name__, exc)

In [ ]:
import random
sets, official_eval = {}, {}

def show(name, examples):
    print("%s: %d examples" % (name, len(examples)))
    if examples:
        ex = random.Random(0).choice(examples)
        print("--- sample prompt ---\n%s\n--- labels %s | target %s\n" % (ex.prompt, ex.labels, [round(t, 2) for t in ex.target]))

try:
    sets["SimBench"] = loaders.parse_simbench(loaders.fetch_simbench())
    show("SimBench", sets["SimBench"])
except Exception as exc:
    print("SimBench skipped:", type(exc).__name__, exc)

In [ ]:
try:
    train_rows, eval_rows = loaders.fetch_subpop()
    sets["SubPOP"] = loaders.parse_subpop(train_rows)
    official_eval["SubPOP"] = loaders.parse_subpop(eval_rows)       # the General Social Survey questions, a different source
    show("SubPOP (train)", sets["SubPOP"]); print("SubPOP official eval:", len(official_eval["SubPOP"]), "examples")
except Exception as exc:
    print("SubPOP skipped (gated: log in and accept the terms on the dataset page):", type(exc).__name__, exc)

In [ ]:
try:
    sets["GlobalOpinionQA"] = loaders.parse_globalopinionqa(loaders.fetch_globalopinionqa())
    show("GlobalOpinionQA", sets["GlobalOpinionQA"])
except Exception as exc:
    print("GlobalOpinionQA skipped:", type(exc).__name__, exc)

In [ ]:
try:
    twin_people, twin_questions = loaders.parse_twin(loaders.fetch_twin())
    print("Twin-2K-500: %d people, %d distinct single-choice questions" % (len(twin_people), len(twin_questions)))
    sets["Twin-2K-500"] = loaders.person_examples(twin_people, twin_questions, "Twin-2K-500",
                                                  "A survey of US adults.", INDIVIDUAL_EXAMPLES, seed=SEED)
    show("Twin-2K-500", sets["Twin-2K-500"])
except Exception as exc:
    print("Twin-2K-500 skipped:", type(exc).__name__, exc)

In [ ]:
for cfg, path, ov in ((loaders.WVS_CFG, WVS_PATH, DEMO_OVERRIDE["WVS"]), (loaders.AFROBAROMETER_CFG, AFROBAROMETER_PATH, DEMO_OVERRIDE["Afrobarometer"])):
    try:
        people, questions = loaders.load_survey_people(path, cfg, max_people=30000, seed=SEED, demo_override=ov)
        sets[cfg["name"]] = loaders.person_examples(people, questions, cfg["name"], cfg["desc"], INDIVIDUAL_EXAMPLES, seed=SEED)
        show(cfg["name"], sets[cfg["name"]])
    except Exception as exc:
        print("%s skipped: %s: %s" % (cfg["name"], type(exc).__name__, exc))

## 6. Assemble train and held-out sets

Held-out data is whole questions or whole respondents, never a random slice of rows. SubPOP uses its official held-out file. **Every dataset is in the training set**, including WVS.

In [ ]:
from survey_ft import build
train_examples, held_examples = build.assemble(
    sets, official_eval=official_eval, eval_fraction=EVAL_FRACTION,
    max_train_per_dataset=MAX_TRAIN_PER_DATASET, eval_cap_per_source=EVAL_CAP_PER_SOURCE, seed=SEED)
build.summary(train_examples, held_examples)

## 7. Load Qwen3.5 with Unsloth and check the adapters

The cell prints the adapters per module type. With `LORA_TARGETS = "auto"` it **stops if no `linear_attn.*` adapter attached**, because that would silently be the first run's coverage again. Expected at rank 16: 18 adapters on each of the five `linear_attn` types, 24 on each MLP type, 6 on each attention type, 10,822,656 trainable parameters.

`before` is the held-out error of the untrained model.

In [ ]:
from survey_ft import train
model, tok = train.load_model(MODEL_NAME, max_seq_len=MAX_LEN, backend=BACKEND, lora_r=LORA_R, lora_alpha=LORA_ALPHA, targets=LORA_TARGETS)
kinds, trainable = train.describe_adapters(model)
print(kinds, "| trainable parameters:", trainable)
if LORA_TARGETS == "auto" and not any(k.startswith("linear_attn") for k in kinds):
    raise RuntimeError("No linear_attn adapters attached: Unsloth's finetune_* filters kept them out. "
                       "Try BACKEND='transformers', or set LORA_TARGETS='no_linear_attn' to accept the first run's coverage.")
rows_train = train.prepare(train_examples, tok, weights=SOURCE_WEIGHTS)
rows_held = train.prepare(held_examples, tok)
before = train.evaluate(model, tok, rows_held, max_len=MAX_LEN)
import pandas as pd
pd.DataFrame(before).T.round(3)

## 8. Train

The step time is measured first (a forward and backward pass, no optimiser step), multiplied by `STEP_TIME_FACTOR`, and the budget is turned into a step count. Check the printed estimate before you leave it running. With `SAVE_STEPS` the adapter is checkpointed into `CKPT_DIR`; after a crash, run sections 1-7 again, set `RESUME = True` in the settings, and re-run this cell: it continues from the latest checkpoint (the step count is recomputed from the budget, so use the same `TRAIN_BUDGET_S` or set `MAX_STEPS`).

In [ ]:
import time
sps = train.seconds_per_step(model, tok, rows_train, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, max_len=MAX_LEN)
steps = MAX_STEPS or max(1, int(TRAIN_BUDGET_S / (sps * STEP_TIME_FACTOR)))
print("measured %.1f s/step x %.2f -> %d steps (~%.1f h); %d of %d training examples seen (%.2f epochs)" %
      (sps, STEP_TIME_FACTOR, steps, steps * sps * STEP_TIME_FACTOR / 3600, steps * BATCH_SIZE * GRAD_ACCUM,
       len(rows_train), steps * BATCH_SIZE * GRAD_ACCUM / len(rows_train)))

In [ ]:
t0 = time.time()
result = train.train(model, tok, rows_train, max_len=MAX_LEN, epochs=1.0, max_steps=steps,
                     batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, lr=LR, logging_steps=10,
                     output_dir=CKPT_DIR, save_steps=SAVE_STEPS, resume=True if RESUME else None)
print(result.metrics)
print("took %.1f min" % ((time.time() - t0) / 60))

## 9. Did it help? Held-out error, before and after

In [ ]:
after = train.evaluate(model, tok, rows_held, max_len=MAX_LEN)
b, a = pd.DataFrame(before).T, pd.DataFrame(after).T
report = pd.concat({"before": b[["kl", "w1", "top1"]], "after": a[["kl", "w1", "top1"]]}, axis=1)
report[("change", "kl")] = a["kl"] - b["kl"]
report[("uniform", "kl")] = a["uniform_kl"]
report.round(3)

Read it per source. For group-level sets a good result is KL below the uniform baseline and falling. For individual-level sets the target is one person's answer, so the KL is the log loss and even a good model stays well above zero; compare against `uniform` and `before`, not zero. Held-out sources are whole questions or respondents, so a gain is generalisation, not memorisation.

## 10. Save and check the adapter

The LoRA adapter is small. It goes in the competition zip next to `main.py` (`lora/adapter_config.json` and `lora/adapter_model.safetensors`); `models.txt` names the base model, `Qwen/Qwen3.5-0.8B`. The check below counts what was saved: the competition's `main.py` folds every `lora_A` / `lora_B` pair into the base weights, `linear_attn.*` included.

In [ ]:
model.save_pretrained("qwen35_survey_lora")
tok.save_pretrained("qwen35_survey_lora")
import json
from safetensors import safe_open
cfg = json.load(open("qwen35_survey_lora/adapter_config.json"))
with safe_open("qwen35_survey_lora/adapter_model.safetensors", "pt") as fh:
    keys = list(fh.keys())
a_keys = [k for k in keys if ".lora_A." in k]
print("r / alpha:", cfg.get("r"), "/", cfg.get("lora_alpha"), "| rslora:", cfg.get("use_rslora"), "| dora:", cfg.get("use_dora"))
print("lora_A tensors: %d (of which linear_attn: %d)" % (len(a_keys), sum("linear_attn" in k for k in a_keys)))
print("size: %.1f MB" % (os.path.getsize("qwen35_survey_lora/adapter_model.safetensors") / 1e6))
assert len(a_keys) == len([k for k in keys if ".lora_B." in k]), "lora_A / lora_B counts differ"

In [ ]:
# Package the folder for download. On Colab this offers the zip as a download; on Kaggle it is already in /kaggle/working.
!zip -r qwen_lora_v1_final.zip qwen35_survey_lora
try:
    from google.colab import files
    files.download('qwen_lora_v1_final.zip')
except ImportError:
    print("not on Colab: find qwen_lora_v1_final.zip in the working directory")